In [16]:
import pandas as pd
import os.path as op
import platform
import glob
import os
from scipy.io import loadmat
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import bambi
# import pingouin

In [46]:
sns.lmplot?


Signature:
sns.lmplot(
    data=None,
    *,
    x=None,
    y=None,
    hue=None,
    col=None,
    row=None,
    palette=None,
    col_wrap=None,
    height=5,
    aspect=1,
    markers='o',
    sharex=None,
    sharey=None,
    hue_order=None,
    col_order=None,
    row_order=None,
    legend=True,
    legend_out=None,
    x_estimator=None,
    x_bins=None,
    x_ci='ci',
    scatter=True,
    fit_reg=True,
    ci=95,
    n_boot=1000,
    units=None,
    seed=None,
    order=1,
    logistic=False,
    lowess=False,
    robust=False,
    logx=False,
    x_partial=None,
    y_partial=None,
    truncate=True,
    x_jitter=None,
    y_jitter=None,
    scatter_kws=None,
    line_kws=None,
    facet_kws=None,
)
Docstring:
Plot data and regression model fits across a FacetGrid.

This function combines :func:`regplot` and :class:`FacetGrid`. It is
intended as a convenient interface to fit regression models across
conditional subsets of a dataset.

When thinking about how to assign variable

In [17]:
# Importing the files with reaction time and with the fitted parameters

if platform.system() == "Windows":
    wanted_dir = "/Volumes/SDrive/multlearn-sns/Modelling/Fitting/bestFittingVals"
else:
    wanted_dir = "/Volumes/SDrive/multlearn-sns/Modelling/Fitting/bestFittingVals"

V0_files = [file for file in glob.glob(wanted_dir + '/**/V0*', recursive=True)]
V1_files = [file for file in glob.glob(wanted_dir + '/**/V1*', recursive=True)]
spe_files = [file for file in glob.glob(wanted_dir + '/**/spe.npy', recursive=True)]

if platform.system() == "Windows":
    wanted_dir = "/data/sourcedata/behavior/modified_files"
else:
    wanted_dir = "/Volumes/SDrive/data/sourcedata/behavior/modified_files"

files = [os.path.join(root, name) for root, dirs, files in os.walk(wanted_dir) for name in files if name.endswith('savedValues.csv')]

files = files[:-1]
data_folder = '/Volumes/SDrive/data/'
# files


In [25]:
# Creating dataFrame with relevant indices and fitted values and RT. 
subjects = []
Vdiff_abs_data = []
Vdiff_data = []
Vsum_data = []
spe_data = []

for i in range(len(spe_files)):
    spe = np.load(spe_files[i])
    spe_data.append(spe)
    V0 = np.load(V0_files[i])
    V1 = np.load(V1_files[i])
    # V1_data.append(V0)
    Vdiff = (V1 - V0)[:, :-1]
    # Vdiff = (np.abs(V0-V1))[:,:-1]
    Vdiff_abs = np.abs(Vdiff)    
    Vdiff_data.append(Vdiff)
    Vdiff_abs_data.append(Vdiff_abs)
    Vsum = (V0+V1)[:,:-1]
    Vsum_data.append(Vsum)
    subjects.append(int(spe_files[i][-10:-8]))
    
# Flatten the data lists
Vdiff_data = np.array(Vdiff_data).flatten()
Vdiff_abs_data = np.array(Vdiff_abs_data).flatten()
Vsum_data = np.array(Vsum_data).flatten()
spe_data = np.array(spe_data).flatten()

# Create the index values for runNumber, trialNumber, and subject
subject = np.repeat(subjects, 6 * 60)
runNumber = np.repeat(np.arange(1, 7), 60 * 58)
trialNumber = np.tile(np.arange(1, 61), 6 * 58)

# Create the DataFrame
df = pd.DataFrame({
    'Vdiff_abs': Vdiff_abs_data,
    'Vdiff': Vdiff_data,    
    'Vsum': Vsum_data,
    'spe': spe_data,
    'runNumber': runNumber,
    'trialNumber': trialNumber,
    'subject': subject
})

# # Set the desired columns as the DataFrame's index
df.set_index(['runNumber', 'trialNumber', 'subject'], inplace=True)

RTdataset = []
for subject in subjects:
    file_path = op.join(data_folder, 'sourcedata', 'behavior', 'modified_files', f'modified_participant{subject:02d}_savedValues.csv')
    
    subject_rows = df.index.get_level_values('subject') == subject

    for key in ['responseTime', 'visual', 'audio', 'tactile', 'action']:
        # Read the CSV file and extract the 'responseTime' column
        data = pd.read_csv(file_path)[key]

        # Create the index values for the subject's rows in the DataFrame
        
        # Assign the 'responseTime' values to the corresponding subject in the DataFrame
        df.loc[subject_rows, key] = data.values


In [26]:
from scipy.stats import zscore

In [27]:
df['spe_z'] = df.groupby('subject', group_keys=False)['spe'].apply(zscore)
df['Vdiff_abs_z'] = df.groupby('subject', group_keys=False)['Vdiff_abs'].apply(zscore)
df['Vsum_z'] = df.groupby('subject', group_keys=False)['Vsum'].apply(zscore)

In [31]:
df.isnull().any(0)

/var/folders/fg/ps8b9dqs3b3cgp4jb85dxwfnsyv51d/T/ipykernel_86331/3511071586.py:1: FutureWarning: In a future version of pandas all arguments of DataFrame.any and Series.any will be keyword-only.
  df.isnull().any(0)


Vdiff_abs       False
Vdiff           False
Vsum            False
spe             False
responseTime     True
visual          False
audio            True
tactile          True
action           True
spe_z           False
Vdiff_abs_z     False
Vsum_z          False
dtype: bool

In [32]:
import statsmodels.api as sm

df = df[~df['responseTime'].isnull()]

# Assuming df is your DataFrame with the specified structure
# Extract the predictor variables and response variable
# df = df.dropna(axis=0)
X = df[['Vdiff_abs_z', 'Vsum_z', 'spe_z']]
y = df['responseTime']

# Standardize the predictor variables
X = (X - X.mean()) / X.std()

# Fit the random effects model with increased iterations
model = sm.MixedLM(y, X, groups=df.index.get_level_values('subject'))
result = model.fit()


# Print the model summary
print(result.summary())


          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: responseTime
No. Observations: 20685   Method:             REML        
No. Groups:       58      Scale:              0.1664      
Min. group size:  343     Log-Likelihood:     -11070.3964 
Max. group size:  360     Converged:          Yes         
Mean group size:  356.6                                   
----------------------------------------------------------
                Coef.  Std.Err.   z    P>|z| [0.025 0.975]
----------------------------------------------------------
Vdiff_abs_z     -0.010    0.003 -3.600 0.000 -0.016 -0.005
Vsum_z          -0.004    0.003 -1.383 0.167 -0.010  0.002
spe_z            0.021    0.003  7.555 0.000  0.016  0.027
Group Var        3.172    1.436                           



In [33]:
df

Vdiff_abs     Vdiff      Vsum       spe  \
runNumber trialNumber subject                                            
1         1           1         0.000000  0.000000  1.000000  2.197225   
          2           1         0.269841  0.269841  0.730159  2.302585   
          3           1         0.454979 -0.454979  0.545021  2.397895   
          4           1         0.251575 -0.251575  1.251575  2.484907   
          5           1         0.418066 -0.418066  0.581934  2.564949   
...                                  ...       ...       ...       ...   
6         56          64        0.098595  0.098595  1.031358  3.060271   
          57          64        0.031358  0.031358  1.035879  1.871802   
          58          64        0.215475 -0.215475  1.148238  2.397895   
          59          64        0.066963  0.066963  0.999726  2.258782   
          60          64        0.153020  0.153020  1.035433  2.273598   

                               responseTime  visual  audio  tactile  action  \
runNumber trialNumber subject                                                 
1         1           1            2.226177     0.0    NaN      2.0     0.0   
          2           1            1.879317     1.0    NaN      1.0     1.0   
          3           1            2.399425     2.0    NaN      2.0     0.0   
          4           1            2.111797     0.0    NaN      0.0     1.0   
          5           1            1.813504     2.0    NaN      1.0     0.0   
...                                     ...     ...    ...      ...     ...   
6         56          64           1.466607     0.0    1.0      NaN     1.0   
          57          64           1.213081     1.0    0.0      NaN     0.0   
          58          64           1.370891     2.0    2.0      NaN     1.0   
          59          64           1.331330     1.0    1.0      NaN     1.0   
          60          64           1.032242     2.0    2.0      NaN     1.0   

                                  spe_z  Vdiff_abs_z    Vsum_z  
runNumber trialNumber subject                                   
1         1           1       -0.145829    -2.077096 -0.169363  
          2           1        0.137445    -0.933496 -0.889115  
          3           1        0.393698    -0.148869 -1.382939  
          4           1        0.627638    -1.010909  0.501667  
          5           1        0.842842    -0.305310 -1.284479  
...                                 ...          ...       ...  
6         56          64       2.134963    -0.793638  0.005250  
          57          64      -1.002173    -1.034114  0.020648  
          58          64       0.386527    -0.375613  0.403344  
          59          64       0.019318    -0.906771 -0.102489  
          60          64       0.058425    -0.598988  0.019130  

[20685 rows x 12 columns]

In [34]:
bambi.Model?

Init signature:
bambi.Model(
    formula,
    data,
    family='gaussian',
    priors=None,
    link=None,
    categorical=None,
    potentials=None,
    dropna=False,
    auto_scale=True,
    noncentered=True,
    extra_namespace=None,
)
Docstring:     
Specification of model class.

Parameters
----------
formula : str or bambi.formula.Formula
    A model description written using the formula syntax from the ``formulae`` library.
data : pandas.DataFrame
    A pandas dataframe containing the data on which the model will be fit, with column
    names matching variables defined in the formula.
family : str or bambi.families.Family
    A specification of the model family (analogous to the family object in R). Either
    a string, or an instance of class ``bambi.families.Family``. If a string is passed, a
    family with the corresponding name must be defined in the defaults loaded at ``Model``
    initialization. Valid pre-defined families are ``"bernoulli"``, ``"beta"``,
    ``"binomial"

In [38]:
df_audio = df[~df.audio.isnull()]

In [42]:
df_audio['spe'].isnull().sum()

0

In [44]:
resp_model_audio = bambi.Model('action ~ Vdiff + Vdiff:spe + (Vdiff + Vdiff:spe|subject)', 
                               df_audio.iloc[:1000].reset_index(),
                               family='bernoulli', link='probit')

In [45]:
idata_resp_audio = resp_model_audio.fit()

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: InplaceDimShuffle{}(TensorConstant{(1,) of 6})
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1918, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1078, in transform
    return self.fn(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/tensor/rewriting/basic.py", line 1138, in constant_folding
    thunk = node.op.make_thunk(node, storage_map, compute_map, no_recycling=[])
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/link/c/op.py", line 131, in ma


You can find the C code in this temporary file: /var/folders/fg/ps8b9dqs3b3cgp4jb85dxwfnsyv51d/T/pytensor_compilation_error_e6d2iunq


ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: InplaceDimShuffle{}(TensorConstant{(1,) of 6})
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1918, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1078, in transform
    return self.fn(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/tensor/rewriting/basic.py", line 1138, in constant_folding
    thunk = node.op.make_thunk(node, storage_map, compute_map, no_recycling=[])
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/link/c/op.py", line 131, in ma


You can find the C code in this temporary file: /var/folders/fg/ps8b9dqs3b3cgp4jb85dxwfnsyv51d/T/pytensor_compilation_error_n_8_mfaa


ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: InplaceDimShuffle{}(TensorConstant{(1,) of 6})
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1918, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1078, in transform
    return self.fn(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/tensor/rewriting/basic.py", line 1138, in constant_folding
    thunk = node.op.make_thunk(node, storage_map, compute_map, no_recycling=[])
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/link/c/op.py", line 131, in ma

...

ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: InplaceDimShuffle{}(TensorConstant{(1,) of 6})
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1918, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1078, in transform
    return self.fn(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/tensor/rewriting/basic.py", line 1138, in constant_folding
    thunk = node.op.make_thunk(node, storage_map, compute_map, no_recycling=[])
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/link/c/op.py", line 131, in ma


You can find the C code in this temporary file: /var/folders/fg/ps8b9dqs3b3cgp4jb85dxwfnsyv51d/T/pytensor_compilation_error_9xp5274m


ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: InplaceDimShuffle{}(TensorConstant{(1,) of 6})
ERROR (pytensor.graph.rewriting.basic): TRACEBACK:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1918, in process_node
    replacements = node_rewriter.transform(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/graph/rewriting/basic.py", line 1078, in transform
    return self.fn(fgraph, node)
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/tensor/rewriting/basic.py", line 1138, in constant_folding
    thunk = node.op.make_thunk(node, storage_map, compute_map, no_recycling=[])
  File "/Users/sbedi/mambaforge/envs/multlearn/lib/python3.9/site-packages/pytensor/link/c/op.py", line 131, in ma


You can find the C code in this temporary file: /var/folders/fg/ps8b9dqs3b3cgp4jb85dxwfnsyv51d/T/pytensor_compilation_error_t7rklt_o


KeyboardInterrupt: 